In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["OPENBLAS_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"

In [2]:
import pandas as pd 
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

In [17]:
from xgboost import XGBClassifier

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, recall_score, accuracy_score, precision_score

In [23]:
df = pd.read_csv("~/mission_sih/mission_sih/data/type_3_dataset.csv")

In [24]:

df["category"].value_counts(dropna=False)
#df.info()

category
industrial_facility    8572
quarry                   71
gas_flare                64
power_plant              37
Name: count, dtype: int64

In [25]:
df.drop(columns="acq_date", inplace=True)

In [26]:
df = df.replace({"category":{"industrial_facility":0, "quarry":1, "gas_flare":2, "power_plant":3}})
df.head()

,latitude,longitude,brightness,scan,track,acq_time,confidence,bright_t31,frp,daynight,type,match_dist_m,category
0,22.79488,86.20175,303.88,0.39,0.36,1956,1,291.97,1.12,1,3,823.705235,0
1,22.79219,86.19742,309.66,0.39,0.36,1956,1,294.21,1.04,1,3,821.842283,0
2,21.18458,81.38937,312.06,0.49,0.41,1956,1,297.98,1.56,1,3,106.958488,0
3,11.63223,92.75088,336.30,0.38,0.36,705,1,303.66,3.53,0,3,305.551027,1
4,23.55042,87.24222,308.04,0.46,0.39,1937,1,294.05,1.31,1,3,661.567717,0


In [27]:
scaler = StandardScaler()

df_scaled = scaler.fit_transform(df)

In [28]:
df["category"] = df['category'].astype(int)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8744 entries, 0 to 8743
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   latitude      8744 non-null   float64
 1   longitude     8744 non-null   float64
 2   brightness    8744 non-null   float64
 3   scan          8744 non-null   float64
 4   track         8744 non-null   float64
 5   acq_time      8744 non-null   int64  
 6   confidence    8744 non-null   int64  
 7   bright_t31    8744 non-null   float64
 8   frp           8744 non-null   float64
 9   daynight      8744 non-null   int64  
 10  type          8744 non-null   int64  
 11  match_dist_m  8744 non-null   float64
 12  category      8744 non-null   int64  
dtypes: float64(8), int64(5)
memory usage: 888.2 KB


In [29]:
X = df.drop(columns="category")
y = df["category"]

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [31]:
y_train.head(10)

2002    0
4645    0
1916    0
1550    0
2358    0
6727    0
3886    0
7948    0
1739    0
7145    0
Name: category, dtype: int64

In [ ]:
xgb = XGBClassifier(
    booster="gbtree", ############
    n_jobs=-1,
    verbosity=2,
    random_state=42,
    validate_parameters=True,
    n_estimators=100, ############
    max_depth=3, #########
    max_leaves=0,
    min_child_weight=1,
    gamma=0.1,                   
    grow_policy="lossguide",    
    tree_method="exact",       
)

In [33]:
model = xgb.fit(X_train, y_train)

[21:22:53] INFO: /__w/xgboost/xgboost/src/tree/updater_prune.cc:144: tree pruning end, 10 extra nodes, 0 pruned nodes, max_depth=3
[21:22:53] INFO: /__w/xgboost/xgboost/src/tree/updater_prune.cc:144: tree pruning end, 6 extra nodes, 0 pruned nodes, max_depth=3
[21:22:53] INFO: /__w/xgboost/xgboost/src/tree/updater_prune.cc:144: tree pruning end, 4 extra nodes, 0 pruned nodes, max_depth=2
[21:22:53] INFO: /__w/xgboost/xgboost/src/tree/updater_prune.cc:144: tree pruning end, 8 extra nodes, 0 pruned nodes, max_depth=3
[21:22:53] INFO: /__w/xgboost/xgboost/src/tree/updater_prune.cc:144: tree pruning end, 8 extra nodes, 0 pruned nodes, max_depth=3
[21:22:53] INFO: /__w/xgboost/xgboost/src/tree/updater_prune.cc:144: tree pruning end, 8 extra nodes, 0 pruned nodes, max_depth=3
[21:22:53] INFO: /__w/xgboost/xgboost/src/tree/updater_prune.cc:144: tree pruning end, 4 extra nodes, 0 pruned nodes, max_depth=2
[21:22:53] INFO: /__w/xgboost/xgboost/src/tree/updater_prune.cc:144: tree pruning end, 6 

In [34]:
y_pred = model.predict(X_test)

In [35]:
confusion_matrix(y_test, y_pred)

array([[1709,    0,    0,    0],
       [   5,   14,    0,    0],
       [   0,    0,   12,    0],
       [   5,    0,    0,    4]])

In [36]:
print(f1_score(y_test, y_pred, average="micro"))
print(f1_score(y_test, y_pred, average="weighted"))

0.9942824471126358
0.9935244479267967


In [37]:
print(recall_score(y_test, y_pred, average="micro"))
print(recall_score(y_test, y_pred, average="weighted"))

0.9942824471126358
0.9942824471126358


In [38]:
print(precision_score(y_test, y_pred, average="micro"))
print(precision_score(y_test, y_pred, average="weighted"))

0.9942824471126358
0.9943157080369368


In [39]:
model.score(X_test, y_test)

0.9942824471126358

In [40]:
import joblib
joblib.dump(model, "hf_model/type_3_model.joblib")

['hf_model/type_3_model.joblib']

In [41]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [42]:
hf_token = os.getenv("HUGGING_FACE_API")

In [43]:
from huggingface_hub import HfApi, create_repo

# Your Hugging Face repo
REPO_ID = "bazik-0/mission-sih-type-3"


with open("hf_model/model_predict.py", "w") as f:
    f.write("""
import joblib
import pandas as pd

model = joblib.load("type_3_model.joblib")

FEATURES = [
    "latitude",
    "longitude",
    "scan",
    "track",
    "track_scan",
    "frp",
    "radiation",
    "brightness",
    "bright_t31",
    "final_bright",
    "acq_time",
    "daynight",
    "confidence",
    "type",
    "match_dist_m"
]

def predict(data):
    X = pd.DataFrame([data])[FEATURES]
    return model.predict(X)[0]

def predict_proba(data):
    X = pd.DataFrame([data])[FEATURES]
    return model.predict_proba(X)[0].tolist()
""")

# 3. Requirements
with open("hf_model/requirements.txt", "w") as f:
    f.write("""scikit-learn
pandas
numpy
joblib
""")

In [44]:

api = HfApi(token=hf_token)

# 4. Create HF repo
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True
)

RepoUrl('https://huggingface.co/bazik-0/mission-sih-type-3', endpoint='https://huggingface.co', repo_type='model', repo_id='bazik-0/mission-sih-type-3')

In [46]:
api.upload_folder(
    folder_path="hf_model",
    repo_id=REPO_ID,
    repo_type="model",
    ignore_patterns=[
        "model.joblib",
        "type_0_model.joblib",    
        "type_2_model.joblib" 
    ]
)

print("Uploaded successfully!")
print(f"https://huggingface.co/{REPO_ID}")

Uploaded successfully!
https://huggingface.co/bazik-0/mission-sih-type-3
